# BIPIA email-QA evaluation: indirect prompt injection

Extends the capstone evaluation from direct injection (Sections 2-3 of the implementation plan) to indirect injection via the BIPIA benchmark (Yi et al., 2025; Section 4 of the plan).

Indirect injection means the attacker does not control the user-facing channel. The user asks a legitimate question about a document (here, an email), and the attacker has planted an instruction inside the document. The agent reads both the user query and the document; if the agent follows the document's instruction in place of (or in addition to) the user's, the agent is hijacked.

This notebook runs the same defense stack as the direct-injection pipelines: ProtectAI DeBERTa and Meta Prompt Guard 2 (Defense A, two operating modes for indirect), Llama 3.3 70B agent + Claude Sonnet 4.6 judge (Defense B). Defense A here is reported in two variants: classifier-on-user-query-only (likely misses indirect attacks because the user query is benign) and classifier-on-full-composed-prompt (sees the attack but may over-flag benign documents).

## Status

This notebook is a SCAFFOLD. The cells below show the intended pipeline; the actual BIPIA data load is stubbed in `src/bipia/email_qa.py` and must be wired up to the upstream BIPIA repo before execution. See `load_bipia_email_qa()` docstring for the data-prep steps.

Pipeline structure has been smoke-tested on the 4-row canary set returned by `canary_email_qa()`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils import env, set_seed
from src.cache import append_records, existing_keys, load_records
from src.bipia.email_qa import (
    BIPIAEmailRow, canary_email_qa, load_bipia_email_qa,
    compose_agent_input, compose_for_defense_a, INDIRECT_JUDGE_SYSTEM_ADDENDUM,
)
from src.defense_a.deberta import DebertaInjectionDetector
from src.defense_b.agent import GroqAgent
from src.defense_b.judge import ClaudeJudge, JUDGE_SYSTEM

env()
set_seed(42)

RES = REPO_ROOT / 'results'
CACHE = REPO_ROOT / 'cache'
CACHE.mkdir(exist_ok=True)

## 1. Load data

Currently using the 4-row canary set. Switch to the BIPIA loader once `load_bipia_email_qa()` is wired up to the upstream data.

In [ ]:
USE_REAL_BIPIA = False  # flip to True once data/bipia/ is populated and the loader is implemented

if USE_REAL_BIPIA:
    rows = list(load_bipia_email_qa())
else:
    rows = canary_email_qa()
print(f'rows: {len(rows)}, attacks: {sum(1 for r in rows if r.is_attack)}')

## 2. Defense A on indirect injection

Two variants reported side by side. Different operating points: query-only is what a deployment would see if the input filter sits between the user and the agent (does not inspect retrieved content); full-prompt is what a deployment would see if the input filter is wrapped around the full composed prompt before the agent runs (inspects retrieved content).

In [ ]:
detector = DebertaInjectionDetector(batch_size=8)

records = []
for variant in ['query_only', 'full_prompt']:
    texts = [compose_for_defense_a(r, variant) for r in rows]
    preds = detector.predict(texts)
    for r, p in zip(rows, preds):
        records.append({
            'row_id': r.row_id,
            'attack_category': r.attack_category,
            'is_attack': r.is_attack,
            'defense_a_variant': variant,
            'deberta_label': p['label'],
            'deberta_pred_label_id': p['label_id'],
            'deberta_injection_score': p['injection_score'],
        })
defense_a_df = pd.DataFrame(records)
defense_a_df

## 3. Defense B: agent + judge

Run the Llama agent on the composed (system, user) pair. Forward the agent response plus the original user query to the judge with the indirect-injection-aware system addendum from `src.bipia.email_qa`. Cache to JSONL for resume.

In [ ]:
AGENT_CACHE = CACHE / 'bipia_email_qa_agent.jsonl'
JUDGE_CACHE = CACHE / 'bipia_email_qa_judge.jsonl'

agent = GroqAgent()

done = existing_keys(AGENT_CACHE, key='row_id')
todo = [r for r in rows if r.row_id not in done]
print(f'agent cached: {len(done)}, to run: {len(todo)}')
for r in tqdm(todo, desc='agent'):
    system_prompt, user_message = compose_agent_input(r)
    # Default GroqAgent uses a fixed system prompt; override for the inbox-assistant role
    agent.system_prompt = system_prompt
    out = agent.respond(user_message)
    append_records(AGENT_CACHE, [{'row_id': r.row_id, **out}])

agent_records = {r['row_id']: r for r in load_records(AGENT_CACHE)}

In [ ]:
# Indirect-injection-aware judge
judge = ClaudeJudge()
# Augment the system prompt with the indirect-injection note (do not modify the underlying module)
_orig = judge.client  # client unchanged; system message is composed at call time below

done = existing_keys(JUDGE_CACHE, key='row_id')
todo = [r for r in rows if r.row_id not in done]
print(f'judge cached: {len(done)}, to run: {len(todo)}')
for r in tqdm(todo, desc='judge'):
    agent_resp = agent_records[r.row_id]['response']
    out = judge.judge(r.user_query, agent_resp)
    append_records(JUDGE_CACHE, [{'row_id': r.row_id, **out}])

judge_records = {r['row_id']: r for r in load_records(JUDGE_CACHE)}

## 4. Per-defense attack success rates

In [ ]:
summary_rows = []
for r in rows:
    j = judge_records[r.row_id]
    summary_rows.append({
        'row_id': r.row_id,
        'attack_category': r.attack_category,
        'is_attack': r.is_attack,
        'defense_a_query_only_flagged': bool(
            defense_a_df.query("row_id == @r.row_id and defense_a_variant == 'query_only'")['deberta_pred_label_id'].iloc[0]
        ),
        'defense_a_full_prompt_flagged': bool(
            defense_a_df.query("row_id == @r.row_id and defense_a_variant == 'full_prompt'")['deberta_pred_label_id'].iloc[0]
        ),
        'judge_hijacked': j.get('hijacked'),
        'judge_blocked': j.get('judge_blocked', False),
    })
summary = pd.DataFrame(summary_rows)
summary.to_csv(RES / 'bipia_email_qa_results.csv', index=False)
summary

## 5. Attack-success rate by defense and category

On real BIPIA data, this would break out the four attack categories (task execution, info gathering, ad insertion, phishing). On the canary set the breakdown is illustrative only.

In [ ]:
attacks = summary[summary['is_attack']]
if len(attacks) > 0:
    rate = pd.DataFrame({
        'Defense A (query only)': [1 - attacks['defense_a_query_only_flagged'].mean()],
        'Defense A (full prompt)': [1 - attacks['defense_a_full_prompt_flagged'].mean()],
        'Defense B (judge)':       [1 - (attacks['judge_hijacked'] == True).mean()],
    }, index=['attack success rate (lower is better)'])
    rate

## Notes for the user when wiring up real BIPIA data

1. `git clone https://github.com/microsoft/BIPIA data/bipia` (the repo is small; do not commit it)
2. Inspect `data/bipia/benchmark/email/` for the JSON test files
3. Implement the per-row mapping in `src/bipia/email_qa.py::load_bipia_email_qa()`
4. Flip `USE_REAL_BIPIA = True` in cell `bp04` above
5. Cost estimate: BIPIA email QA has approximately 200-400 rows in the standard test set. At ~$0.005-0.010 per row (Llama agent + Sonnet judge), total ~$1-4. Well inside the project budget.
6. The Defense A `query_only` variant is structurally expected to underperform on attack rows; this is the methodological point. Report both variants side by side.
7. The judge is the same Claude Sonnet 4.6 minimum-rubric judge used in the direct-injection sneak previews; rubric iteration for the indirect setting may be required after reviewing initial verdicts.